# Plot results for offline prompting in the "dragon" example

In [1]:
%cd ..
%pwd  # should be "llm-adaptation"

C:\Users\micha\OneDrive - Univerzita Karlova\research\2024-LLM-DEECo\llm-adaptation


C:\Users\micha\OneDrive - Univerzita Karlova\research\2024-LLM-DEECo\llm-adaptation\.venv\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


'C:\\Users\\micha\\OneDrive - Univerzita Karlova\\research\\2024-LLM-DEECo\\llm-adaptation'

In [10]:
from pathlib import Path
import pandas as pd
import shutil
import subprocess
import sys
import json

In [3]:
results_folder = Path("generated_adaptations/dragon")

## Run experiments

In [19]:
prompts = {
    "default": "generated_adaptations/prompts/dragon.txt",
    "high": "generated_adaptations/prompts/dragon_strategy.txt",  # high-level strategy
}
repeats = 1
start = 1

In [25]:
# import generated_adaptations.generator as generator

In [20]:
for variant, prompt in prompts.items():
    for repeat in range(start, start + repeats):
        folder_name = f"41mini_{variant}_{repeat:02d}"
        print(f"\n{folder_name}\n")
        folder = results_folder / folder_name
        folder.mkdir(parents=True, exist_ok=True)
        shutil.copy(prompt, folder / "01_01_user.md")
        cmd  = [sys.executable, "generated_adaptations/generator.py", f"--folder={str(folder)}"]  #, "--retries_test=1", "--retries_simulation=0"]
        result = subprocess.run(cmd, capture_output=True, text=True)  # even capture_output=False does not stream in real-time, so we rather capture it and print it later
        print(result.stdout)
        print(result.stderr)
        # generator.main([f"--folder={str(folder)}"])  # this also does not show the real-time output


41mini_default_01

Loaded 2 messages from generated_adaptations\farm\41mini_default_01.
Querying LLM with prompt 01_01_user.
LLM response saved to '01_02_llm.md'.
Code block saved to 'generated_adaptations\farm\41mini_default_01\code_01_02.py'.
TOKENS USED:
Input: 979, Output: 1310 (reasoning: 0)
Response time (seconds): 21.8

Running tests: pytest generated_adaptations/tests -q --tb=short -rA --show-capture=no --color=no --example=farm --adaptation_name=41mini_default_01/code_01_02
Test exit code: 0
Running simulation for 'generated_adaptations\farm\41mini_default_01\code_01_02.py'.
  Run #1/3: python main.py farm/configs/default.yaml generated_adaptations/configs/generated.yaml farm/configs/config_no_battery.yaml DSL/drones.yaml --extra_config={"name": "41mini_default_01/code_01_02", "log_dir.append": "/41mini_default_01/code_01_02", "adaptation_name": "generated_adaptations.farm.41mini_default_01.code_01_02.SmartFarmAdaptation"} -s 1 -e 1
    StdErr: 62 lines
    Component assigned

## Results

In [4]:
folders = list(results_folder.glob("41nano_*"))
# folders = [results_folder / "41mini"]
print([f.stem for f in folders])

['41nano_default', '41nano_default_01', '41nano_default_02']


In [20]:
summary = pd.DataFrame(columns=["llm", "params"])
best = pd.DataFrame(columns=["llm", "params"])
for folder in folders:
    llm, params = folder.stem.split("_", 1) if "_" in folder.stem else (folder.stem, "")
    summary.loc[len(summary), ["llm", "params"]] = [llm, params]
    best.loc[len(best), ["llm", "params"]] = [llm, params]
    for file in (folder / "results").glob("*.txt"):
        name = file.stem.removeprefix("code_")
        if "_test_fail" in name:
            code = name.removesuffix("_test_fail")
            summary.loc[len(summary) - 1, code + "_test"] = "fail"
        elif "_test_pass" in name:
            code = name.removesuffix("_test_pass")
            summary.loc[len(summary) - 1, code + "_test"] = "pass"
        else:
            print(f"Unknown file: {file}")
    for file in (folder / "results").glob("*.json"):
        name = file.stem.removeprefix("code_")
        if "_simulation_result" in name:
            code = name.removesuffix("_simulation_result")
            results = json.load(open(file))
            result = results["winrate"]
            summary.loc[len(summary) - 1, code + "_result"] = result
            best.loc[len(best) - 1, code.split("_")[0] + "_result"] = result  # only take the first part of the code file name
        else:
            print(f"Unknown file: {file}")
summary = summary.astype("object")
best = best.astype("object")
summary.fillna("", inplace=True)
best.fillna("", inplace=True)

In [21]:
summary.set_index(["llm", "params"], inplace=True)
best.set_index(["llm", "params"], inplace=True)

In [22]:
summary = summary.reindex(sorted(summary.columns), axis=1)
best = best.reindex(sorted(best.columns), axis=1)

In [23]:
summary

01_02_result 01_02_test 01_04_result 01_04_test  \
llm    params                                                       
41nano default                                                      
       default_01          0.2       pass                           
       default_02          0.0       fail          0.0       fail   

                  01_06_result 01_06_test 01_08_result 01_08_test  \
llm    params                                                       
41nano default                                                      
       default_01                                                   
       default_02          0.0       fail          0.0       fail   

                  02_02_result 02_02_test 03_02_result 03_02_test  
llm    params                                                      
41nano default                                                     
       default_01          0.2       pass          0.2       pass  
       default_02

In [24]:
best

01_result 02_result 03_result
llm    params                                  
41nano default                                 
       default_01       0.2       0.2       0.2
       default_02       0.0